# Pixels to Predictions: Model 15 Final Run

### Final Strategy
- **Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` only
- **Objective:** multiple-choice contrastive log-likelihood, not generation loss
- **Scoring:** deterministic hybrid of compact choice-label likelihood and short choice-text likelihood
- **Prompt:** metadata + lecture + hint + same-model caption + visual reasoning instructions
- **Adapters:** DoRA when supported, LoRA fallback, attention + MLP targets under 5M trainable parameters
- **Safety:** resumable checkpoints, best/latest adapters, optimizer/scheduler/scaler/RNG state
- **Output:** `submission.csv`


---
## Imports and Offline Setup


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

import ast
import gc
import inspect
import json
import math
import pickle
import random
import shutil
import time
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance

import torch
import torch.nn.functional as F
from torch.optim import AdamW
from tqdm.auto import tqdm

from transformers import AutoModelForVision2Seq, AutoProcessor, get_cosine_schedule_with_warmup
from peft import LoraConfig, PeftModel, get_peft_model


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True

MODEL_NAME = "HuggingFaceTB/SmolVLM-500M-Instruct"
RUN_NAME = "model15_final"
SCORE_MODE = "mean"
PROMPT_STYLE = "visual_metadata_caption_hybrid_choice"
CHOICE_SCORE_STYLE = "hybrid_label_and_choice_text"

LR = 1e-5
WEIGHT_DECAY = 0.01
EPOCHS = 3
GRAD_ACCUM_STEPS = 8
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.05

TARGET_IMAGE_SIZE = 640
MAX_PROMPT_CHARS = 3400
LECTURE_WORDS = 160
ANSWER_MAX_CHARS = 260
LABEL_SCORE_WEIGHT = 0.70
TEXT_SCORE_WEIGHT = 0.30

CAPTION_BATCH_SIZE = 2
CAPTION_MAX_NEW_TOKENS = 32
CAPTION_SAVE_EVERY = 100
CAPTION_PROMPT = "<image>\nDescribe this scientific image briefly and factually in one or two sentences. Caption:"

LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
LORA_R_CANDIDATES = [8, 6, 4, 2]
LORA_ALPHA_FACTOR = 2
LORA_DROPOUT = 0.05
USE_DORA_REQUESTED = True
TRAINABLE_PARAM_LIMIT = 5_000_000
USE_TORCH_COMPILE = False

SAVE_EVERY_OPT_STEPS = 500


def find_project_root():
    candidates = [Path("."), Path(".."), Path("/kaggle/working")]
    for path in candidates:
        if (path / "data" / "train.csv").exists() or (path / "runs").exists() or (path / "experiment_notebooks").exists():
            return path.resolve()
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working")
    return Path(".").resolve()


PROJECT_ROOT = find_project_root()
RUN_DIR = PROJECT_ROOT / "runs" / RUN_NAME
BEST_DIR = RUN_DIR / "best"
LATEST_DIR = RUN_DIR / "latest"
CHECKPOINTS_DIR = RUN_DIR / "checkpoints"
HISTORY_PATH = RUN_DIR / "training_history.csv"
VAL_SCORES_PATH = RUN_DIR / "val_scores.pkl"
TEST_SCORES_PATH = RUN_DIR / "test_scores.pkl"
SUBMISSION_PATH = PROJECT_ROOT / "submission.csv"

for p in [RUN_DIR, BEST_DIR, LATEST_DIR, CHECKPOINTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_DIR:", RUN_DIR)
print("SCORE_MODE:", SCORE_MODE)
print("CHOICE_SCORE_STYLE:", CHOICE_SCORE_STYLE)
print("PROMPT_STYLE:", PROMPT_STYLE)
print("TARGET_IMAGE_SIZE:", TARGET_IMAGE_SIZE)
print("EPOCHS:", EPOCHS)


---
## Data Loading


In [ ]:
def find_data_dir():
    candidates = [
        PROJECT_ROOT / "data",
        Path("../data"),
        Path("data"),
        Path("/kaggle/input/pixels-to-predictions"),
        Path("/kaggle/input/pixels-to-predictions/data"),
    ]
    for path in candidates:
        if (path / "train.csv").exists() and (path / "val.csv").exists() and (path / "test.csv").exists():
            return path
    raise FileNotFoundError("Could not find competition CSV files. Add the Pixels-to-Predictions dataset.")


DATA_DIR = find_data_dir()
print("DATA_DIR:", DATA_DIR)

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)
display(train_df.head(2))


In [ ]:
def parse_choices(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    try:
        return ast.literal_eval(str(x))
    except Exception:
        return json.loads(str(x))


for df in [train_df, val_df, test_df]:
    df["choices_list"] = df["choices"].apply(parse_choices)
    df["num_choices"] = df["choices_list"].apply(len).astype(int)

train_df["answer"] = train_df["answer"].astype(int)
val_df["answer"] = val_df["answer"].astype(int)

print("Choice counts train:")
display(train_df["num_choices"].value_counts().sort_index())
print("Choice counts val:")
display(val_df["num_choices"].value_counts().sort_index())
print("Choice counts test:")
display(test_df["num_choices"].value_counts().sort_index())

assert train_df["answer"].between(0, train_df["num_choices"] - 1).all()
assert val_df["answer"].between(0, val_df["num_choices"] - 1).all()


In [ ]:
def resolve_image_path(row):
    rel = Path(str(row["image_path"]))
    candidates = [
        DATA_DIR / rel,
        DATA_DIR / "images" / rel,
        DATA_DIR / "images" / rel.name,
        PROJECT_ROOT / rel,
        PROJECT_ROOT / "data" / rel,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"Image not found for {row.get('id', 'unknown')}: {row['image_path']}")


def load_image_for_row(row):
    return Image.open(resolve_image_path(row)).convert("RGB")


sample_path = resolve_image_path(train_df.iloc[0])
print("Sample image path:", sample_path)
print("Sample image size:", Image.open(sample_path).size)


---
## Prompt and Image Preparation


In [ ]:
def clean_text(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return " ".join(str(x).replace("\r", " ").replace("\t", " ").split())


def truncate_words(text, max_words):
    words = clean_text(text).split()
    return " ".join(words[:max_words])


def truncate_chars(text, max_chars):
    text = clean_text(text)
    if len(text) <= max_chars:
        return text
    shortened = text[:max_chars].rsplit(" ", 1)[0].strip()
    return shortened if shortened else text[:max_chars]


def get_caption(row):
    return clean_text(row.get("image_caption", ""))


def build_choice_text(choice):
    answer = truncate_chars(choice, ANSWER_MAX_CHARS)
    return answer if answer else "unknown"


def build_prompt(row):
    choices = row["choices_list"]
    choice_lines = "\n".join([f"{i}: {clean_text(choice)}" for i, choice in enumerate(choices)])

    prompt = "<image>\n"
    prompt += "Answer the science multiple-choice question.\n"
    prompt += "Carefully analyze the scientific image.\n"
    prompt += "Pay attention to labels, axes, charts, maps, diagrams, tables, arrows, legends, units, and spatial relationships.\n"
    prompt += "Use both visual evidence and textual context.\n"
    prompt += "Use the task, subject, topic, category, skill, grade, hint, lecture, and generated image caption when helpful.\n"
    prompt += "Reason carefully before selecting the answer.\n"
    prompt += "Return only the number of the correct choice.\n\n"

    fields = [
        ("Task", row.get("task", "")),
        ("Grade", row.get("grade", "")),
        ("Subject", row.get("subject", "")),
        ("Topic", row.get("topic", "")),
        ("Category", row.get("category", "")),
        ("Skill", row.get("skill", "")),
    ]
    for name, value in fields:
        value = clean_text(value)
        if value:
            prompt += f"{name}: {value}\n"

    lecture = truncate_words(row.get("lecture", ""), LECTURE_WORDS)
    if lecture:
        prompt += f"\nLecture:\n{lecture}\n"

    hint = clean_text(row.get("hint", ""))
    if hint:
        prompt += f"\nHint:\n{hint}\n"

    caption = get_caption(row)
    if caption:
        prompt += f"\nGenerated image caption:\n{caption}\n"

    prompt += f"\nQuestion:\n{clean_text(row['question'])}\n\n"
    prompt += f"Answer choices:\n{choice_lines}\n\n"
    prompt += "Correct choice:"

    if len(prompt) > MAX_PROMPT_CHARS:
        keep_tail = prompt[-1200:]
        prompt = prompt[:MAX_PROMPT_CHARS - len(keep_tail) - 20] + "\n...\n" + keep_tail

    return prompt


print(build_prompt(train_df.iloc[0])[:1800])


In [ ]:
def resize_long_side(image, long_side=TARGET_IMAGE_SIZE):
    w, h = image.size
    if max(w, h) == long_side:
        return image
    scale = long_side / max(w, h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    return image.resize((new_w, new_h), Image.Resampling.BICUBIC)


def light_train_augment(image):
    image = resize_long_side(image, TARGET_IMAGE_SIZE)

    if random.random() < 0.55:
        w, h = image.size
        scale = random.uniform(0.97, 1.0)
        crop_w = max(1, int(w * scale))
        crop_h = max(1, int(h * scale))
        left = random.randint(0, max(0, w - crop_w))
        top = random.randint(0, max(0, h - crop_h))
        image = image.crop((left, top, left + crop_w, top + crop_h)).resize((w, h), Image.Resampling.BICUBIC)

    if random.random() < 0.25:
        angle = random.uniform(-1.0, 1.0)
        image = image.rotate(angle, resample=Image.Resampling.BICUBIC, expand=False, fillcolor=(255, 255, 255))

    if random.random() < 0.35:
        factor = random.uniform(0.96, 1.04)
        image = ImageEnhance.Brightness(image).enhance(factor)

    return image


def prepare_image(row, train=False):
    image = load_image_for_row(row)
    if train:
        return light_train_augment(image)
    return resize_long_side(image, TARGET_IMAGE_SIZE)


---
## Model Loading


In [ ]:
def find_model_source():
    candidates = [
        PROJECT_ROOT / "models" / "SmolVLM-500M-Instruct",
        Path("../models/SmolVLM-500M-Instruct"),
        Path("../input/SmolVLM-500M-Instruct"),
        Path("/kaggle/input/smolvlm-500m-instruct"),
        Path("/kaggle/input/smolvlm-500m-instruct/SmolVLM-500M-Instruct"),
        Path("/kaggle/input/huggingfacetb-smolvlm-500m-instruct"),
    ]
    for path in candidates:
        if path.exists() and (path / "config.json").exists():
            print("Using local model path:", path)
            return str(path)

    print("Using Hugging Face model id with local_files_only=True:", MODEL_NAME)
    print("Offline mode is enabled; this will use the local HF cache only and fail if unavailable.")
    return MODEL_NAME


MODEL_SOURCE = find_model_source()

DTYPE = torch.float16
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    DTYPE = torch.bfloat16
print("Model dtype:", DTYPE)


def load_processor():
    processor = AutoProcessor.from_pretrained(MODEL_SOURCE, local_files_only=True)
    if getattr(processor, "tokenizer", None) is not None and processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    return processor


def configure_processor_resolution(processor, target_size=TARGET_IMAGE_SIZE):
    image_processor = getattr(processor, "image_processor", None)
    if image_processor is None:
        return processor

    original_size = getattr(image_processor, "size", None)
    print("Original processor image size:", original_size)

    try:
        if isinstance(image_processor.size, dict):
            size = dict(image_processor.size)
            if "longest_edge" in size:
                size["longest_edge"] = target_size
            if "shortest_edge" in size:
                size["shortest_edge"] = min(size.get("shortest_edge", target_size), target_size)
            if "height" in size:
                size["height"] = target_size
            if "width" in size:
                size["width"] = target_size
            image_processor.size = size
        elif isinstance(image_processor.size, int):
            image_processor.size = target_size
        print("Configured processor image size:", getattr(image_processor, "size", None))
    except Exception as e:
        print("Could not modify processor image size; using PIL resizing only.", repr(e))

    return processor


processor = configure_processor_resolution(load_processor())


In [ ]:
def preferred_attention_implementation():
    return "flash_attention_2" if DEVICE.type == "cuda" else "eager"


def load_base_model(attn_implementation=None):
    attn_implementation = attn_implementation or preferred_attention_implementation()
    kwargs = dict(
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True,
        local_files_only=True,
        _attn_implementation=attn_implementation,
    )

    try:
        loaded_model = AutoModelForVision2Seq.from_pretrained(MODEL_SOURCE, **kwargs)
        print("Attention implementation:", attn_implementation)
    except Exception as e:
        if attn_implementation != "eager":
            print("Flash Attention 2 unavailable; falling back to eager attention.", repr(e))
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return load_base_model(attn_implementation="eager")

        print("Could not set eager attention explicitly; retrying with model default.", repr(e))
        kwargs.pop("_attn_implementation", None)
        loaded_model = AutoModelForVision2Seq.from_pretrained(MODEL_SOURCE, **kwargs)
        print("Attention implementation: model default")

    loaded_model.config.use_cache = False
    loaded_model.to(DEVICE)
    return loaded_model


def autocast_context():
    if DEVICE.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=DTYPE)
    return nullcontext()


def move_to_device(inputs):
    return {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in inputs.items()}


---
## Same-Model Caption Cache


In [ ]:
def clean_caption(text):
    text = clean_text(text).replace("Caption:", "").strip()
    if not text:
        return ""
    parts = []
    for chunk in text.replace("!", ".").replace("?", ".").split("."):
        chunk = chunk.strip()
        if chunk:
            parts.append(chunk)
        if len(parts) >= 2:
            break
    caption = ". ".join(parts)
    if caption and not caption.endswith("."):
        caption += "."
    return caption[:300]


def load_caption_cache(path):
    if path.exists():
        with open(path, "rb") as f:
            cache = pickle.load(f)
        print(f"Loaded caption cache: {path} ({len(cache)} captions)")
        return cache
    return {}


def save_caption_cache(path, cache):
    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as f:
        pickle.dump(cache, f)
    tmp.replace(path)


def generate_caption_batch(caption_model, rows):
    images = [prepare_image(row, train=False) for _, row in rows.iterrows()]
    prompts = [CAPTION_PROMPT] * len(images)
    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inputs = move_to_device(inputs)

    with torch.inference_mode(), autocast_context():
        generated = caption_model.generate(
            **inputs,
            max_new_tokens=CAPTION_MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )

    prompt_lens = inputs["attention_mask"].sum(dim=1).detach().cpu().tolist()
    captions = []
    for i, prompt_len in enumerate(prompt_lens):
        new_tokens = generated[i, int(prompt_len):]
        text = processor.tokenizer.decode(new_tokens, skip_special_tokens=True)
        captions.append(clean_caption(text))

    del inputs, generated
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return captions


def ensure_caption_cache(df, split_name, caption_model):
    cache_path = RUN_DIR / f"caption_cache_{split_name}.pkl"
    cache = load_caption_cache(cache_path)
    missing_ids = [rid for rid in df["id"].tolist() if rid not in cache]
    print(f"{split_name}: {len(cache)} cached, {len(missing_ids)} missing")

    if not missing_ids:
        return cache

    caption_model.eval()
    missing_df = df[df["id"].isin(missing_ids)].reset_index(drop=True)

    for start in tqdm(range(0, len(missing_df), CAPTION_BATCH_SIZE), desc=f"caption {split_name}"):
        batch = missing_df.iloc[start:start + CAPTION_BATCH_SIZE]
        captions = generate_caption_batch(caption_model, batch)
        for row_id, caption in zip(batch["id"].tolist(), captions):
            cache[row_id] = caption

        done = start + len(batch)
        if done % CAPTION_SAVE_EVERY == 0 or done == len(missing_df):
            save_caption_cache(cache_path, cache)
            print(f"[Caption Cache Saved] {split_name}: {len(cache)} captions")

    return cache


def attach_captions(df, cache):
    df = df.copy()
    df["image_caption"] = df["id"].map(cache).fillna("")
    return df


In [ ]:
caption_model = load_base_model()
caption_model.eval()

caption_cache_train = ensure_caption_cache(train_df, "train", caption_model)
caption_cache_val = ensure_caption_cache(val_df, "val", caption_model)
caption_cache_test = ensure_caption_cache(test_df, "test", caption_model)

train_df = attach_captions(train_df, caption_cache_train)
val_df = attach_captions(val_df, caption_cache_val)
test_df = attach_captions(test_df, caption_cache_test)

print("Caption example:", train_df.iloc[0]["image_caption"])

del caption_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


---
## DoRA / LoRA Adapter Setup


In [ ]:
def print_trainable_parameters(model):
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    pct = 100 * trainable_params / max(total_params, 1)
    print(f"trainable params: {trainable_params:,} || all params: {total_params:,} || trainable%: {pct:.4f}")
    return trainable_params, total_params


def lora_config_kwargs(r, use_dora):
    kwargs = dict(
        r=r,
        lora_alpha=r * LORA_ALPHA_FACTOR,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=LORA_TARGET_MODULES,
        task_type="CAUSAL_LM",
    )
    if "use_dora" in inspect.signature(LoraConfig.__init__).parameters:
        kwargs["use_dora"] = bool(use_dora)
    elif use_dora:
        print("PEFT does not support use_dora in this environment. Falling back to standard LoRA.")
    return kwargs


def enable_training_memory_savers(model):
    model.config.use_cache = False
    try:
        model.gradient_checkpointing_enable()
        print("Gradient checkpointing enabled.")
    except Exception as e:
        print("Gradient checkpointing could not be enabled:", repr(e))
    try:
        model.enable_input_require_grads()
    except Exception as e:
        print("Input grad enabling skipped:", repr(e))
    return model


def adapter_mode_candidates():
    dora_supported = "use_dora" in inspect.signature(LoraConfig.__init__).parameters
    if USE_DORA_REQUESTED and dora_supported:
        return [True, False]
    if USE_DORA_REQUESTED and not dora_supported:
        print("PEFT does not support DoRA in this environment. Falling back to standard LoRA.")
    return [False]


def maybe_compile_model(model):
    if not USE_TORCH_COMPILE:
        print("torch.compile disabled for final-run checkpoint stability.")
        return model
    if not hasattr(torch, "compile"):
        print("torch.compile unavailable in this PyTorch version.")
        return model
    try:
        print("Compiling model with torch.compile...")
        return torch.compile(model)
    except Exception as e:
        print("torch.compile failed; continuing uncompiled.", repr(e))
        return model


def attach_new_adapter_under_limit():
    last_error = None

    for use_dora in adapter_mode_candidates():
        mode_name = "DoRA" if use_dora else "LoRA"
        for r in LORA_R_CANDIDATES:
            print(f"Trying {mode_name} with r={r}, target_modules={LORA_TARGET_MODULES}")
            base_model = load_base_model()
            for p in base_model.parameters():
                p.requires_grad = False

            try:
                cfg = LoraConfig(**lora_config_kwargs(r, use_dora))
                peft_model = get_peft_model(base_model, cfg)
                peft_model = enable_training_memory_savers(peft_model)
                trainable_count, _ = print_trainable_parameters(peft_model)

                if trainable_count < TRAINABLE_PARAM_LIMIT:
                    print(f"Adapter selected: {mode_name} r={r}")
                    return maybe_compile_model(peft_model), {"r": r, "use_dora": use_dora, "target_modules": LORA_TARGET_MODULES}

                print(f"Trainable params {trainable_count:,} exceed limit; retrying with lower rank.")
                del peft_model, base_model
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except Exception as e:
                last_error = e
                print(f"Adapter attempt failed for {mode_name} r={r}:", repr(e))
                del base_model
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    raise RuntimeError(f"No adapter configuration stayed under {TRAINABLE_PARAM_LIMIT:,} trainable params. Last error: {last_error}")


def load_or_create_train_model():
    if (LATEST_DIR / "adapter_config.json").exists():
        print("[Resuming From Checkpoint] Loading latest adapter from:", LATEST_DIR)
        base_model = load_base_model()
        train_model = PeftModel.from_pretrained(base_model, LATEST_DIR, is_trainable=True)
        train_model = enable_training_memory_savers(train_model)
        trainable_count, _ = print_trainable_parameters(train_model)
        assert trainable_count < TRAINABLE_PARAM_LIMIT, "Trainable parameter limit exceeded."
        return maybe_compile_model(train_model), {"resumed_adapter": str(LATEST_DIR)}

    print("Starting a fresh adapter.")
    train_model, adapter_info = attach_new_adapter_under_limit()
    trainable_count, _ = print_trainable_parameters(train_model)
    assert trainable_count < TRAINABLE_PARAM_LIMIT, "Trainable parameter limit exceeded."
    return train_model, adapter_info


model, adapter_info = load_or_create_train_model()
trainable_param_count, total_param_count = print_trainable_parameters(model)
assert trainable_param_count < 5_000_000, "Trainable parameter limit exceeded."
print("Adapter info:", adapter_info)


---
## Contrastive Multiple-Choice Scoring


In [ ]:
def build_candidate_sets(row):
    choices = row["choices_list"]
    label_candidates = [f" {i}" for i in range(len(choices))]
    text_candidates = [f" {i}: {build_choice_text(choice)}" for i, choice in enumerate(choices)]
    return [label_candidates, text_candidates]


def score_candidate_texts(prompt, image, candidate_texts):
    full_texts = [prompt + candidate for candidate in candidate_texts]

    prompt_inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
        padding=False,
    )
    prompt_len = int(prompt_inputs["input_ids"].shape[1])
    del prompt_inputs

    full_inputs = processor(
        text=full_texts,
        images=[image] * len(candidate_texts),
        return_tensors="pt",
        padding=True,
    )
    full_inputs = move_to_device(full_inputs)

    input_ids = full_inputs["input_ids"]
    attention_mask = full_inputs.get("attention_mask", torch.ones_like(input_ids))

    outputs = model(**full_inputs)
    logits = outputs.logits.float()

    shift_logits = logits[:, :-1, :]
    shift_input_ids = input_ids[:, 1:]
    shift_attention = attention_mask[:, 1:]

    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs.gather(dim=-1, index=shift_input_ids.unsqueeze(-1)).squeeze(-1)

    answer_mask = torch.zeros_like(shift_input_ids, dtype=torch.bool)
    start = max(prompt_len - 1, 0)
    answer_mask[:, start:] = True
    answer_mask = answer_mask & shift_attention.bool()

    scores = []
    for i in range(len(candidate_texts)):
        vals = token_log_probs[i][answer_mask[i]]
        if vals.numel() == 0:
            score = token_log_probs.new_tensor(-1e9)
        elif SCORE_MODE == "mean":
            score = vals.mean()
        elif SCORE_MODE == "sum":
            score = vals.sum()
        else:
            raise ValueError("SCORE_MODE must be 'mean' or 'sum'")
        scores.append(score)

    del outputs, logits, shift_logits, log_probs, token_log_probs, full_inputs
    return torch.stack(scores)


def choice_scores_for_row(row, train=False):
    prompt = build_prompt(row)
    image = prepare_image(row, train=train)
    label_candidates, text_candidates = build_candidate_sets(row)

    all_candidates = label_candidates + text_candidates
    all_scores = score_candidate_texts(prompt, image, all_candidates)
    num_choices = int(row["num_choices"])
    label_scores = all_scores[:num_choices]
    text_scores = all_scores[num_choices:]
    combined_scores = LABEL_SCORE_WEIGHT * label_scores + TEXT_SCORE_WEIGHT * text_scores
    return combined_scores


def contrastive_loss_for_row(row):
    scores = choice_scores_for_row(row, train=True)
    logits = scores.unsqueeze(0)
    target = torch.tensor([int(row["answer"])], dtype=torch.long, device=logits.device)
    loss = F.cross_entropy(logits, target)
    pred = int(torch.argmax(scores.detach()).item())
    return loss, pred, scores


@torch.no_grad()
def predict_row(row):
    model.eval()
    scores = choice_scores_for_row(row, train=False)
    pred = int(torch.argmax(scores).item())
    return pred, scores.detach().cpu().numpy()


In [ ]:
# Quick smoke test before training.
model.eval()
with torch.no_grad(), autocast_context():
    pred, scores = predict_row(val_df.iloc[0])
print("Smoke-test prediction:", pred)
print("Smoke-test scores:", scores)
model.train()


---
## Validation


In [ ]:
def evaluate_model(df=val_df, max_rows=None, save_scores=False):
    eval_df = df.reset_index(drop=True)
    if max_rows is not None:
        eval_df = eval_df.sample(n=max_rows, random_state=SEED).reset_index(drop=True)

    model.eval()
    preds = []
    labels = []
    all_scores = []

    for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="validate"):
        with torch.no_grad(), autocast_context():
            pred, scores = predict_row(row)
        preds.append(pred)
        labels.append(int(row["answer"]))
        all_scores.append(scores)

        if (idx + 1) % 500 == 0:
            running_acc = np.mean(np.array(preds) == np.array(labels))
            print(f"[Validation] {idx + 1}/{len(eval_df)} acc={running_acc:.4f}")

        if torch.cuda.is_available() and (idx + 1) % 250 == 0:
            torch.cuda.empty_cache()

    preds = np.array(preds)
    labels = np.array(labels)
    acc = float(np.mean(preds == labels))

    print(f"Validation accuracy: {acc:.6f}")
    print("Prediction distribution:")
    display(pd.Series(preds).value_counts(normalize=True).sort_index())

    result = {
        "accuracy": acc,
        "preds": preds,
        "labels": labels,
        "scores": all_scores,
        "score_mode": SCORE_MODE,
        "choice_score_style": CHOICE_SCORE_STYLE,
        "prompt_style": PROMPT_STYLE,
    }

    if save_scores:
        with open(VAL_SCORES_PATH, "wb") as f:
            pickle.dump(result, f)
        print("Saved validation scores to:", VAL_SCORES_PATH)

    model.train()
    return result


---
## Checkpoint Safety


In [ ]:
def get_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def set_rng_state(state):
    if not state:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and "cuda" in state:
        torch.cuda.set_rng_state_all(state["cuda"])


def save_checkpoint(adapter_dir, training_state, optimizer=None, scheduler=None, scaler=None):
    adapter_dir = Path(adapter_dir)
    adapter_dir.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(adapter_dir)
    processor.save_pretrained(adapter_dir)

    state = dict(training_state)
    if optimizer is not None:
        state["optimizer_state"] = optimizer.state_dict()
    if scheduler is not None:
        state["scheduler_state"] = scheduler.state_dict()
    if scaler is not None:
        state["scaler_state"] = scaler.state_dict()
    state["rng_state"] = get_rng_state()
    state["adapter_info"] = adapter_info
    state["target_image_size"] = TARGET_IMAGE_SIZE
    state["choice_score_style"] = CHOICE_SCORE_STYLE

    torch.save(state, adapter_dir / "training_state.pt")
    print(f"[Checkpoint Saved] {adapter_dir}")


def safe_torch_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def load_training_state(path):
    state_path = Path(path) / "training_state.pt"
    if state_path.exists():
        print("[Resuming From Checkpoint] Loading training state:", state_path)
        return safe_torch_load(state_path, map_location="cpu")
    print("No training state found. Starting from epoch 0.")
    return None


def checkpoint_step_dir(global_step):
    return CHECKPOINTS_DIR / f"step_{global_step:06d}"


---
## Optimizer and Resume State


In [ ]:
def build_optimizer_and_scheduler(total_epochs):
    trainable_params_local = [p for p in model.parameters() if p.requires_grad]
    optimizer_local = AdamW(trainable_params_local, lr=LR, weight_decay=WEIGHT_DECAY)
    total_optimizer_steps_local = math.ceil(len(train_df) * total_epochs / GRAD_ACCUM_STEPS)
    warmup_steps_local = int(total_optimizer_steps_local * WARMUP_RATIO)
    scheduler_local = get_cosine_schedule_with_warmup(
        optimizer_local,
        num_warmup_steps=warmup_steps_local,
        num_training_steps=total_optimizer_steps_local,
    )
    return trainable_params_local, optimizer_local, scheduler_local, total_optimizer_steps_local, warmup_steps_local


trainable_params, optimizer, scheduler, total_optimizer_steps, warmup_steps = build_optimizer_and_scheduler(EPOCHS)

use_scaler = DEVICE.type == "cuda" and DTYPE == torch.float16
scaler = torch.cuda.amp.GradScaler(enabled=use_scaler)

state = load_training_state(LATEST_DIR)
if state is not None:
    optimizer.load_state_dict(state.get("optimizer_state", optimizer.state_dict()))
    scheduler.load_state_dict(state.get("scheduler_state", scheduler.state_dict()))
    if state.get("scaler_state") is not None:
        scaler.load_state_dict(state["scaler_state"])
    set_rng_state(state.get("rng_state"))
    start_epoch = int(state.get("epoch", 0))
    start_row = int(state.get("next_row", 0))
    global_step = int(state.get("global_step", 0))
    best_val_acc = float(state.get("best_val_acc", -1.0))
    history = list(state.get("history", []))
else:
    start_epoch = 0
    start_row = 0
    global_step = 0
    best_val_acc = -1.0
    history = []

print("total_optimizer_steps:", total_optimizer_steps)
print("warmup_steps:", warmup_steps)
print("start_epoch:", start_epoch)
print("start_row:", start_row)
print("global_step:", global_step)
print("best_val_acc:", best_val_acc)


---
## Training Loop


In [ ]:
def train_one_epoch(epoch, start_row=0):
    model.train()
    epoch_df = train_df.sample(frac=1.0, random_state=SEED + epoch).reset_index(drop=True)

    total_loss = 0.0
    total_correct = 0
    total_count = 0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(range(start_row, len(epoch_df)), desc=f"epoch {epoch + 1}/{EPOCHS}")
    global global_step

    for row_idx in pbar:
        row = epoch_df.iloc[row_idx]

        with autocast_context():
            loss, pred, scores = contrastive_loss_for_row(row)
            loss_to_backprop = loss / GRAD_ACCUM_STEPS

        if use_scaler:
            scaler.scale(loss_to_backprop).backward()
        else:
            loss_to_backprop.backward()

        label = int(row["answer"])
        total_loss += float(loss.detach().cpu())
        total_correct += int(pred == label)
        total_count += 1

        is_update_step = ((row_idx + 1) % GRAD_ACCUM_STEPS == 0) or (row_idx + 1 == len(epoch_df))
        if is_update_step:
            if use_scaler:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, MAX_GRAD_NORM)

            if use_scaler:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            if global_step % SAVE_EVERY_OPT_STEPS == 0 or global_step % 1000 == 0:
                training_state = {
                    "epoch": epoch,
                    "next_row": row_idx + 1,
                    "global_step": global_step,
                    "best_val_acc": best_val_acc,
                    "history": history,
                    "last_train_loss": total_loss / max(total_count, 1),
                    "last_train_acc": total_correct / max(total_count, 1),
                }
                save_checkpoint(checkpoint_step_dir(global_step), training_state, optimizer, scheduler, scaler)
                save_checkpoint(LATEST_DIR, training_state, optimizer, scheduler, scaler)

            if torch.cuda.is_available() and global_step % 50 == 0:
                torch.cuda.empty_cache()

        if total_count > 0:
            pbar.set_postfix({
                "loss": f"{total_loss / total_count:.4f}",
                "acc": f"{total_correct / total_count:.4f}",
                "opt_step": global_step,
            })

        del loss, loss_to_backprop, scores

    return {
        "train_loss": total_loss / max(total_count, 1),
        "train_acc": total_correct / max(total_count, 1),
        "seen_rows": total_count,
    }


In [ ]:
def run_training_loop(target_epochs):
    global start_epoch, start_row, state, best_val_acc, history

    print("=== FINAL TRAINING RUN ===")
    print("Rows:", len(train_df), "train /", len(val_df), "val")
    print("Epochs target:", target_epochs)
    print("LR:", LR)
    print("GRAD_ACCUM_STEPS:", GRAD_ACCUM_STEPS)
    print("MAX_GRAD_NORM:", MAX_GRAD_NORM)
    print("Adapter:", adapter_info)

    for epoch in range(start_epoch, target_epochs):
        print(f"\n=== Epoch {epoch + 1}/{target_epochs} ===")

        if start_row < len(train_df):
            train_metrics = train_one_epoch(epoch, start_row=start_row)
        else:
            print("Training already completed for this epoch; proceeding to validation.")
            train_metrics = state or {}

        pre_val_state = {
            "epoch": epoch,
            "next_row": len(train_df),
            "global_step": global_step,
            "best_val_acc": best_val_acc,
            "history": history,
            "train_metrics": train_metrics,
        }
        save_checkpoint(CHECKPOINTS_DIR / f"epoch_{epoch + 1:02d}_pre_validation", pre_val_state, optimizer, scheduler, scaler)
        save_checkpoint(LATEST_DIR, pre_val_state, optimizer, scheduler, scaler)

        print("\n=== VALIDATION ===")
        val_result = evaluate_model(val_df, save_scores=True)
        val_acc = float(val_result["accuracy"])

        is_best = val_acc > best_val_acc
        if is_best:
            best_val_acc = val_acc
            print(f"[Best Model Updated] val_acc={best_val_acc:.6f}")

        row_history = {
            "epoch": epoch + 1,
            "global_step": global_step,
            "train_loss": train_metrics.get("train_loss", np.nan),
            "train_acc": train_metrics.get("train_acc", np.nan),
            "val_acc": val_acc,
            "best_val_acc": best_val_acc,
            "saved_best": is_best,
        }
        history.append(row_history)
        pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
        display(pd.DataFrame(history).tail())

        post_val_state = {
            "epoch": epoch + 1,
            "next_row": 0,
            "global_step": global_step,
            "best_val_acc": best_val_acc,
            "history": history,
            "train_metrics": train_metrics,
            "val_acc": val_acc,
        }
        save_checkpoint(CHECKPOINTS_DIR / f"epoch_{epoch + 1:02d}_post_validation", post_val_state, optimizer, scheduler, scaler)
        save_checkpoint(LATEST_DIR, post_val_state, optimizer, scheduler, scaler)

        if is_best:
            save_checkpoint(BEST_DIR, post_val_state, optimizer, scheduler, scaler)

        start_row = 0
        start_epoch = epoch + 1
        state = post_val_state

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("\n=== TRAINING COMPLETE ===")
    print(f"Best validation accuracy: {best_val_acc:.6f}")
    print("Best checkpoint:", BEST_DIR)
    print("Latest checkpoint:", LATEST_DIR)
    return best_val_acc


best_val_acc = run_training_loop(EPOCHS)


---
## Load Best Checkpoint


In [ ]:
def load_best_model_for_inference():
    global model, processor
    if not (BEST_DIR / "adapter_config.json").exists():
        print("BEST checkpoint not found; using LATEST checkpoint instead.")
        adapter_dir = LATEST_DIR
    else:
        adapter_dir = BEST_DIR

    try:
        del model
    except NameError:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    processor = configure_processor_resolution(AutoProcessor.from_pretrained(adapter_dir, local_files_only=True))
    base_model = load_base_model()
    model = PeftModel.from_pretrained(base_model, adapter_dir, is_trainable=False)
    model.to(DEVICE)
    model.eval()
    print("Loaded adapter for inference:", adapter_dir)
    return adapter_dir


BEST_USED_DIR = load_best_model_for_inference()


---
## Best Checkpoint Validation


In [ ]:
best_val_result = evaluate_model(val_df, save_scores=True)
print("Best checkpoint validation accuracy:", best_val_result["accuracy"])


---
## Test Inference and Submission


In [ ]:
def predict_test_df(test_df):
    model.eval()
    preds = []
    test_scores = []

    for idx, row in tqdm(test_df.reset_index(drop=True).iterrows(), total=len(test_df), desc="test inference"):
        with torch.no_grad(), autocast_context():
            pred, scores = predict_row(row)

        preds.append(pred)
        test_scores.append({
            "id": row["id"],
            "answer": pred,
            "scores": scores,
            "num_choices": int(row["num_choices"]),
        })

        if (idx + 1) % 500 == 0:
            print(f"[Test] {idx + 1}/{len(test_df)}")
        if torch.cuda.is_available() and (idx + 1) % 250 == 0:
            torch.cuda.empty_cache()

    with open(TEST_SCORES_PATH, "wb") as f:
        pickle.dump(test_scores, f)
    print("Saved test scores to:", TEST_SCORES_PATH)

    return preds


test_preds = predict_test_df(test_df)
submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": test_preds,
})

submission.to_csv(SUBMISSION_PATH, index=False)

print("Saved submission to:", SUBMISSION_PATH)
print(submission.shape)
display(submission.head())
display(submission["answer"].value_counts(normalize=True).sort_index())
